# ChronoLogic: Temporal Image Sequence Ordering
## Research Notebook — Executive Summary, Experiments & Future Directions

---

## 1. Executive Summary

**ChronoLogic** is a vision-language benchmark and algorithm suite for *unsupervised temporal ordering* of image sequences. Given a shuffled set of frames from a real-world process (plant growth, DIY construction, time-lapse), the goal is to recover the correct chronological order without any supervision signal.

### Problem Statement
Recovering temporal order from a disordered collection of images is fundamental to applications ranging from visual story understanding to surgical training video analysis. Classical frame-ordering methods rely on optical flow or supervised temporal models; this project explores purely *embedding-space* approaches using a frozen vision-language encoder (OpenCLIP).

### Dataset
Seven manually curated photo sequences across three content domains:

| Sequence | Domain | Difficulty | Type | Frames |
|---|---|---|---|---|
| Sequence 1 | DIY | Easy | Procedural | 8 |
| Sequence 2 | Plant Growth | Medium | Time-lapse | 8 |
| Sequence 3 | Construction | Hard | Time-lapse | 8 |
| Sequence 4 | Plant Growth | Medium | Time-lapse | 8 |
| Sequence 5 | DIY | Easy | Procedural | 8 |
| Sequence 6 | DIY | Hard | Procedural | 8 |
| Sequence 7 | Coffee | Easy | Procedural | 8 |

### Key Results (Summary)
- **Greedy Nearest Neighbor (GNN)** is the clear best performing method: **61.2% pairwise accuracy**, Kendall τ = **+0.22**
- **Continuity Exhaustive Search** underperforms random at **29.1%** — indicating the continuity objective is a poor proxy for temporal ordering in this dataset
- The **forward/reverse ambiguity** is a persistent structural challenge: all 7 sequences score identically in both orientations (tie)
- **Sequence 4 (Plant Growth)** is the only sequence where GNN achieves a perfect match (Kendall τ = 1.0)
- Error taxonomy: ~57% of predictions from all non-random methods fall in the "scrambled" category — indicating global ordering confusion rather than local swaps

### Engineering Contributions
1. Modular `chronologic` Python package with clearly separated concerns (ordering, evaluation, diagnostics)
2. Cached OpenCLIP embeddings (RN50 + OpenAI weights) to avoid re-computation
3. Fully reproducible benchmark with seeded shuffles per sequence
4. Rich diagnostic pipeline: error taxonomy, trajectory analysis, endpoint distinctiveness, and forward/reverse disambiguation

---
## 2. Setup & Imports

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT = Path(".")  # repo root
DATA = ROOT / "Data"
ANALYSIS = DATA / "analysis"
ORDERING = ANALYSIS / "ordering"
SIMILARITY = ANALYSIS / "similarity"
DIAGNOSTICS = ORDERING / "diagnostics"
EMBEDDINGS = DATA / "embeddings" / "openclip"
MANIFEST = DATA / "manifests" / "sequences.json"

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.15)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlepad"] = 12
PALETTE = {"random": "#9e9e9e", "greedy_nearest_neighbor": "#1976d2", "continuity": "#e65100"}

print("Environment ready.")

---
## 3. Dataset Overview

The benchmark uses 7 image sequences, each with 8 frames representing a real-world temporal process. Frames are shuffled with a fixed per-sequence seed so experiments are reproducible.

In [ ]:
with open(MANIFEST) as f:
    sequences = json.load(f)

manifest_df = pd.DataFrame([
    {
        "Sequence": s["sequence_id"].replace("_", " ").title(),
        "Category": s["category"],
        "Difficulty": s["difficulty"].title(),
        "Type": s["sequence_type"].replace("_", "-").title(),
        "Frames": s["num_frames"],
        "Caption": s["caption"],
    }
    for s in sequences
])

manifest_df.style.set_caption("Dataset Manifest").hide(axis="index").set_properties(**{"text-align": "left"})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Difficulty distribution
diff_counts = manifest_df["Difficulty"].value_counts()
axes[0].bar(diff_counts.index, diff_counts.values, color=["#4caf50", "#ff9800", "#f44336"])
axes[0].set_title("Sequences by Difficulty")
axes[0].set_ylabel("Count")

# Category distribution
cat_counts = manifest_df["Category"].value_counts()
axes[1].bar(cat_counts.index, cat_counts.values, color=sns.color_palette("Set2", len(cat_counts)))
axes[1].set_title("Sequences by Category")
axes[1].tick_params(axis="x", rotation=20)

# Sequence type
type_counts = manifest_df["Type"].value_counts()
axes[2].pie(type_counts.values, labels=type_counts.index, autopct="%1.0f%%",
            colors=["#5c6bc0", "#26c6da"])
axes[2].set_title("Sequence Type Split")

plt.suptitle("Dataset Composition", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Embedding Pipeline

### Architecture
All frames are encoded with **OpenCLIP (RN50, openai weights)** — a contrastively-trained vision-language model. Critically, the encoder is used *frozen*: no fine-tuning is performed. This tests whether off-the-shelf vision-language representations contain sufficient temporal geometry for ordering.

### Why OpenCLIP?
- Joint image-text embedding space allows text-guided direction scoring (start/middle/end prompts)
- Pre-trained on massive internet-scale data with broad visual coverage
- Deterministic and cache-friendly: each sequence's embeddings saved as a `.npy` file

### Workflow
```
Image Frames → OpenCLIP Encoder (RN50, frozen) → 1024-d embeddings
                                                        ↓
                                    Cosine Similarity Matrix (8×8 per sequence)
                                                        ↓
                              Ordering Algorithms    Diagnostics
```

In [ ]:
# Load cached embeddings and show shapes
embed_info = []
for seq in sequences:
    sid = seq["sequence_id"]
    path = EMBEDDINGS / f"{sid}.npy"
    if path.exists():
        arr = np.load(path)
        embed_info.append({"Sequence": sid, "Shape": str(arr.shape), "Dtype": str(arr.dtype),
                           "Norm (mean)": f"{np.linalg.norm(arr, axis=1).mean():.3f}"})

pd.DataFrame(embed_info).style.hide(axis="index").set_caption("Cached OpenCLIP Embeddings")

In [ ]:
# Visualise embedding norms across sequences
fig, ax = plt.subplots(figsize=(10, 4))
for seq in sequences:
    sid = seq["sequence_id"]
    path = EMBEDDINGS / f"{sid}.npy"
    if path.exists():
        arr = np.load(path)
        norms = np.linalg.norm(arr, axis=1)
        ax.plot(range(1, len(norms)+1), norms, marker="o", label=sid.replace("_", " ").title())

ax.set_xlabel("Frame Index")
ax.set_ylabel("L2 Norm of Embedding")
ax.set_title("OpenCLIP Embedding Norms per Frame (Ground-Truth Order)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Similarity Analysis

For each sequence we compute the pairwise cosine similarity matrix over frames in ground-truth order. Two diagnostic statistics characterise the temporal structure:

- **Near-mean**: average similarity of adjacent frame pairs (|i - j| = 1)
- **Far-mean**: average similarity of distant frame pairs (|i - j| ≥ 4)
- **Temporal Contrast Score (TCS)**: `near_mean − far_mean` — higher = stronger temporal signal

A high TCS means nearby frames look alike and distant frames look different, which is the property that makes greedy nearest-neighbour ordering tractable.

In [ ]:
results_df = pd.read_csv(ORDERING / "ordering_results.csv")
by_method = pd.read_csv(ORDERING / "ordering_summary_by_method.csv")
by_seq = pd.read_csv(ORDERING / "ordering_summary_by_sequence.csv")

# Extract one row per sequence for similarity stats (same across all methods)
sim_df = (
    results_df.drop_duplicates(subset="sequence_id")[["sequence_id", "category", "difficulty",
                                                       "sequence_type", "temporal_near_mean",
                                                       "temporal_far_mean", "temporal_contrast_score"]]
    .reset_index(drop=True)
)

sim_df.columns = ["Sequence", "Category", "Difficulty", "Type", "Near Mean", "Far Mean", "TCS"]
sim_df[["Near Mean", "Far Mean", "TCS"]] = sim_df[["Near Mean", "Far Mean", "TCS"]].round(4)
sim_df.style.hide(axis="index") \
    .background_gradient(subset=["TCS"], cmap="YlOrRd") \
    .set_caption("Per-Sequence Temporal Contrast Scores")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Near vs Far scatter with TCS as size
scatter_ax = axes[0]
tcs_raw = results_df.drop_duplicates(subset="sequence_id")
colors = ["#4caf50" if d == "easy" else ("#ff9800" if d == "medium" else "#f44336")
          for d in tcs_raw["difficulty"]]
sc = scatter_ax.scatter(
    tcs_raw["temporal_near_mean"], tcs_raw["temporal_far_mean"],
    c=colors, s=tcs_raw["temporal_contrast_score"] * 2000,
    alpha=0.85, edgecolors="#222", linewidths=0.8
)
for _, row in tcs_raw.iterrows():
    scatter_ax.annotate(row["sequence_id"].replace("sequence_", "S"),
                        (row["temporal_near_mean"], row["temporal_far_mean"]),
                        xytext=(4, 4), textcoords="offset points", fontsize=9)
scatter_ax.set_xlabel("Near-Frame Mean Similarity")
scatter_ax.set_ylabel("Far-Frame Mean Similarity")
scatter_ax.set_title("Near vs. Far Similarity\n(bubble size = TCS)")
from matplotlib.lines import Line2D
legend_els = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=10, label=l)
              for c, l in zip(["#4caf50", "#ff9800", "#f44336"], ["Easy", "Medium", "Hard"])]
scatter_ax.legend(handles=legend_els, title="Difficulty")

# TCS bar chart
bar_ax = axes[1]
bar_data = tcs_raw.sort_values("temporal_contrast_score", ascending=False)
bar_ax.barh(bar_data["sequence_id"].str.replace("sequence_", "S"),
            bar_data["temporal_contrast_score"],
            color=sns.color_palette("RdYlGn", len(bar_data))[::-1])
bar_ax.set_xlabel("Temporal Contrast Score")
bar_ax.set_title("Temporal Contrast Score per Sequence\n(higher = stronger temporal signal)")
bar_ax.axvline(bar_data["temporal_contrast_score"].mean(), ls="--", color="#555", label="Mean")
bar_ax.legend()

plt.tight_layout()
plt.show()

print(f"\nMean TCS: {bar_data['temporal_contrast_score'].mean():.4f}")
print(f"Range:    [{bar_data['temporal_contrast_score'].min():.4f}, {bar_data['temporal_contrast_score'].max():.4f}]")

In [ ]:
# Visualise similarity heatmap for Sequence 2 (highest TCS) if the file exists
heatmap_path = SIMILARITY / "sequence_2_heatmap.png"
if heatmap_path.exists():
    fig, ax = plt.subplots(figsize=(6, 5))
    img = mpimg.imread(str(heatmap_path))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Sequence 2 — Pairwise Cosine Similarity Heatmap (Plant Growth)")
    plt.tight_layout()
    plt.show()
else:
    # Build similarity matrix from cached embeddings and plot inline
    emb = np.load(EMBEDDINGS / "sequence_2.npy")
    normed = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
    sim_mat = normed @ normed.T

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(sim_mat, annot=True, fmt=".2f", cmap="viridis", ax=ax,
                xticklabels=[f"F{i}" for i in range(8)],
                yticklabels=[f"F{i}" for i in range(8)])
    ax.set_title("Sequence 2 — Pairwise Cosine Similarity (Plant Growth)")
    plt.tight_layout()
    plt.show()

---
## 6. Ordering Experiments

Three ordering methods were evaluated, spanning two paradigms:

### 6.1 Random Baseline
Draws a uniformly random permutation of frames using a fixed per-sequence seed. No embedding information is used. **Expected pairwise accuracy ≈ 50%** for 8 frames. This establishes the lower bound.

### 6.2 Greedy Nearest Neighbour (GNN)
Builds a Hamiltonian path through the embedding similarity graph using a greedy forward strategy. Starting from every possible first frame, the algorithm always visits the most similar unvisited frame next. The path with the highest total adjacency score is returned.

**Key property**: optimises `path_score = Σ sim(f_i, f_{i+1})` — a proxy for visual smoothness over time.

### 6.3 Continuity Exhaustive Search
Exhaustively enumerates all 8! = 40,320 permutations and ranks them by a composite continuity score:

$$\text{score} = w_{adj} \cdot \text{adjacency} + w_{cont} \cdot \text{continuity\_penalty} + w_{dir} \cdot \text{direction} + w_{ep} \cdot \text{endpoint}$$

In the base configuration `w_adj = w_cont = 1.0`, `w_dir = w_ep = 0.0`. The exhaustive search guarantees the globally optimal solution under this score.

### 6.4 Ablations (Advanced)
Additional variants were explored in post-hoc diagnostics:
- **continuity_plus_direction** — adds text-direction scoring (`w_dir = 0.75`) from start/middle/end CLIP prompts
- **continuity_plus_endpoint** — adds endpoint distinctiveness weighting (`w_ep = 0.75`)
- **continuity_plus_reverse_disambiguation** — applies forward/reverse tiebreaking after continuity ranking

In [ ]:
# Pretty-print all ordering predictions with scores
pred_df = pd.read_csv(DIAGNOSTICS / "predictions.csv")
pred_df["predicted_order"] = pred_df["predicted_order"].apply(
    lambda x: " → ".join([str(int(i)+1) for i in x.split()])
)
pred_df["score"] = pred_df["score"].apply(lambda x: f"{float(x):.3f}" if str(x) != "nan" else "—")
pred_df.columns = ["Sequence", "Method", "Predicted Order (1-indexed)", "Path Score"]
pred_df.style.hide(axis="index") \
    .set_caption("All Method Predictions (Ground Truth: Frame 1 → 2 → … → 8)")

In [ ]:
# Visualise path scores (GNN vs Continuity vs Random)
path_df = results_df[results_df["path_score"].notna()].copy()
path_df["seq_label"] = path_df["sequence_id"].str.replace("sequence_", "S")

fig, ax = plt.subplots(figsize=(11, 5))
methods = ["greedy_nearest_neighbor", "continuity"]
group_width = 0.35
x = np.arange(7)

for i, method in enumerate(methods):
    sub = path_df[path_df["method"] == method].sort_values("sequence_id")
    ax.bar(x + i * group_width - group_width/2,
           sub["path_score"].values, group_width,
           label=method.replace("_", " ").title(),
           color=list(PALETTE.values())[i+1], alpha=0.88)

ax.set_xticks(x)
ax.set_xticklabels([f"S{i+1}" for i in range(7)])
ax.set_ylabel("Path Score (Σ adjacent cosine sim)")
ax.set_title("Greedy Nearest Neighbor vs. Continuity — Path Scores per Sequence")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Evaluation Metrics

Four complementary metrics are reported for each (sequence, method) pair:

| Metric | Formula | Range | Interpretation |
|---|---|---|---|
| **Exact Match Accuracy** | 1 if `pred == truth` | {0, 1} | Only 1 when order is perfect |
| **Pairwise Order Accuracy** | Fraction of concordant $(i,j)$ pairs | [0, 1] | Random baseline = 0.5 |
| **Normalized Kendall Agreement** | `1 - inversions / max_inversions` | [0, 1] | Random baseline = 0.5 |
| **Kendall τ** | `1 - 2·inversions / max_inversions` | [-1, 1] | Random = 0, perfect = +1, reversed = −1 |

> **Note:** Pairwise Accuracy and Normalized Kendall Agreement are identical in implementation — both measure the proportion of correctly ordered pairs.

In [ ]:
# Aggregate method summary
by_method_disp = by_method.copy()
by_method_disp.columns = ["Method", "Exact Match", "Pairwise Acc",
                           "Norm. Kendall", "Kendall τ", "Mean Inversions"]
for col in ["Exact Match", "Pairwise Acc", "Norm. Kendall", "Kendall τ"]:
    by_method_disp[col] = by_method_disp[col].round(4)
by_method_disp["Mean Inversions"] = by_method_disp["Mean Inversions"].round(2)
by_method_disp["Method"] = by_method_disp["Method"].str.replace("_", " ").str.title()

by_method_disp.style.hide(axis="index") \
    .background_gradient(subset=["Pairwise Acc", "Kendall τ"], cmap="RdYlGn") \
    .background_gradient(subset=["Mean Inversions"], cmap="RdYlGn_r") \
    .set_caption("Method Summary — Averaged Across All 7 Sequences")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metric_cols = ["pairwise_order_accuracy", "kendall_tau"]
metric_labels = ["Pairwise Order Accuracy", "Kendall τ"]
method_order = ["random", "continuity", "greedy_nearest_neighbor"]
method_labels = ["Random", "Continuity", "Greedy NN"]

for ax, col, label in zip(axes, metric_cols, metric_labels):
    values = [by_method.set_index("method").loc[m, col] if m in by_method["method"].values else 0
              for m in method_order]
    bars = ax.bar(method_labels, values,
                  color=[PALETTE[m] for m in method_order], alpha=0.9, width=0.5)
    ax.axhline(0.5, ls="--", color="#888", label="Random baseline (0.5)")
    ax.axhline(0.0, ls=":", color="#aaa")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
                f"{val:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_ylim(-0.6 if col == "kendall_tau" else 0, 1.05)
    ax.set_title(label)
    ax.set_ylabel(label)
    if col == "pairwise_order_accuracy":
        ax.legend()

plt.suptitle("Method Comparison — Averaged Over All 7 Sequences", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Per-sequence heatmap: kendall_tau by method
pivot_tau = results_df.pivot_table(
    index="sequence_id", columns="method", values="kendall_tau"
).reindex(columns=method_order)
pivot_tau.index = pivot_tau.index.str.replace("sequence_", "S")
pivot_tau.columns = method_labels

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot_tau, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, linecolor="#eee")
ax.set_title("Kendall τ per Sequence × Method\n(green = better ordering, red = reversed/wrong)")
ax.set_ylabel("Sequence")
plt.tight_layout()
plt.show()

In [ ]:
# Inversion count distribution
fig, ax = plt.subplots(figsize=(10, 4))
max_possible_inversions = (8 * 7) // 2  # = 28 for n=8

for method, label in zip(method_order, method_labels):
    sub = results_df[results_df["method"] == method]
    ax.plot(sub["sequence_id"].str.replace("sequence_", "S"),
            sub["inversion_count"], marker="o", label=label,
            color=PALETTE[method], linewidth=2, markersize=8)

ax.axhline(max_possible_inversions / 2, ls="--", color="#888", alpha=0.7, label="Expected random (14)")
ax.axhline(max_possible_inversions, ls=":", color="#f44336", alpha=0.5, label="Worst case (28)")
ax.set_ylabel("Inversion Count")
ax.set_title("Inversion Count per Sequence (lower = better, max = 28 for n=8)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Diagnostics & Analysis

Beyond headline metrics, four diagnostic modules were run to understand *why* methods succeed or fail.

### 8.1 Error Taxonomy

Each prediction is categorised into one of five failure modes:

| Category | Definition |
|---|---|
| **exact** | Perfect match with ground truth |
| **reversed** | Prediction is the exact reverse of ground truth |
| **local_swap** | Only one adjacent pair swapped |
| **endpoint_error** | Only the first or last frame is wrong |
| **scrambled** | None of the above — global confusion |

In [ ]:
taxonomy_summary = pd.read_csv(DIAGNOSTICS / "error_taxonomy" / "taxonomy_summary.csv")
taxonomy_rows = pd.read_csv(DIAGNOSTICS / "error_taxonomy" / "taxonomy_rows.csv")

# Pivot for plotting
tax_pivot = taxonomy_summary.pivot_table(
    index="taxonomy", columns="method", values="fraction", aggfunc="first"
).reindex(columns=method_order)
tax_pivot.columns = method_labels
tax_pivot = tax_pivot.fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
tax_pivot.T.plot(kind="bar", ax=ax, colormap="Set2", width=0.7, edgecolor="#333", linewidth=0.5)
ax.set_ylabel("Fraction of Sequences")
ax.set_title("Error Taxonomy by Method")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title="Error Type", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

print("\nRaw taxonomy summary:")
taxonomy_summary.pivot_table(index="method", columns="taxonomy", values="fraction").round(3)

**Observation:** The "scrambled" category dominates all methods. This tells us that the failures are not systematic (no method consistently produces exact reversals or simple swaps) — they are caused by fundamental confusion in embedding space, where no clear temporal gradient exists.

### 8.2 Forward / Reverse Disambiguation

The **forward/reverse problem** is a near-universal challenge in temporal ordering: if a sequence looks visually smooth in both directions (e.g., a plant growing vs. a plant dying), the ordering score is symmetric by construction.

In [ ]:
fwd_rev = pd.read_csv(DIAGNOSTICS / "forward_reverse" / "forward_reverse_scores.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.bar(fwd_rev["sequence_id"].str.replace("sequence_", "S"),
       fwd_rev["forward_minus_reverse"], color="#5c6bc0", alpha=0.85)
ax.axhline(0, color="#f44336", ls="--", linewidth=1.5, label="Zero gap (tie)")
ax.set_ylabel("Forward − Reverse Score")
ax.set_title("Forward vs. Reverse Score Gap (Continuity Ordering)")
ax.legend()

ax2 = axes[1]
winners = fwd_rev["winner"].value_counts()
ax2.pie(winners.values, labels=winners.index, autopct="%1.0f%%",
        colors=["#ef5350" if l == "reverse" else ("#66bb6a" if l == "forward" else "#bdbdbd")
                for l in winners.index])
ax2.set_title("Resolution Outcome Across All Sequences")

plt.suptitle("Forward / Reverse Disambiguation Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nConclusion: ALL 7 sequences score identically in forward and reverse (\'tie\').")
print("The continuity score has zero discriminative power for orientation.")

### 8.3 Endpoint Distinctiveness

A well-structured temporal sequence should have **edge frames** (first and last) that are visually distinct from the middle — high distance to the embedding centroid. This property aids in identifying anchors for ordering.

In [ ]:
endpoint_df = pd.read_csv(DIAGNOSTICS / "endpoint" / "endpoint_distinctiveness.csv")

fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
axes = axes.flatten()

for i, (seq_id, grp) in enumerate(endpoint_df.groupby("sequence_id")):
    ax = axes[i]
    colors = ["#f44336" if ep else "#1976d2" for ep in grp["is_true_endpoint"]]
    ax.bar(grp["frame_index"], grp["distance_to_centroid"], color=colors, alpha=0.85)
    ax.set_title(seq_id.replace("sequence_", "Seq "), fontsize=10)
    ax.set_xlabel("Frame")
    ax.set_ylabel("Distance to Centroid")
    ax.set_xticks(grp["frame_index"])

# Legend
from matplotlib.patches import Patch
legend_els = [Patch(facecolor='#f44336', label='True Endpoint'), Patch(facecolor='#1976d2', label='Middle Frame')]
axes[6].legend(handles=legend_els, loc="center", fontsize=12)
axes[6].axis("off")
axes[7].axis("off")

plt.suptitle("Endpoint Distinctiveness — Distance to Embedding Centroid\n(red = true first/last frame)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Quantify: are true endpoints actually the most distinctive frames?
endpoint_summary = []
for seq_id, grp in endpoint_df.groupby("sequence_id"):
    true_ep_dist = grp[grp["is_true_endpoint"] == 1]["distance_to_centroid"].mean()
    middle_dist = grp[grp["is_true_endpoint"] == 0]["distance_to_centroid"].mean()
    max_dist_frame = grp.loc[grp["distance_to_centroid"].idxmax(), "frame_index"]
    is_endpoint_most_distinct = max_dist_frame in grp[grp["is_true_endpoint"]==1]["frame_index"].values
    endpoint_summary.append({
        "Sequence": seq_id.replace("sequence_", "S"),
        "Avg Endpoint Dist": round(true_ep_dist, 4),
        "Avg Middle Dist": round(middle_dist, 4),
        "Endpoint > Middle?": "✓" if true_ep_dist > middle_dist else "✗",
        "Most Distinct Frame is Endpoint?": "✓" if is_endpoint_most_distinct else "✗",
    })

ep_sum_df = pd.DataFrame(endpoint_summary)
ep_sum_df.style.hide(axis="index").set_caption("Endpoint Distinctiveness Summary")

### 8.4 Pairwise Error Analysis

In [ ]:
pairwise_errors_path = DIAGNOSTICS / "pairwise_errors" / "pairwise_error_rows.csv"
if pairwise_errors_path.exists():
    pw_df = pd.read_csv(pairwise_errors_path)
    
    # Confusion heatmap — which frame positions are most frequently misordered?
    if {"left_item", "right_item", "method"}.issubset(pw_df.columns):
        for method in method_order[1:]:  # skip random
            sub = pw_df[(pw_df["method"] == method)]
            if sub.empty:
                continue
            error_pairs = sub[sub.get("is_correct", sub.get("correct", pd.Series(True, index=sub.index))) == False]
            print(f"{method}: {len(error_pairs)} misordered pairs logged")
    else:
        print("Pairwise error file columns:", pw_df.columns.tolist())
        print(pw_df.head())
else:
    print("Pairwise error rows not found; run scripts/run_diagnostics.py to regenerate.")

---
## 9. Sequence-Level Deep Dive

Sequence 4 (Plant Growth, strawberry seed) is the **only perfect prediction** by GNN. Let's understand why it works there and not elsewhere.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sequence 4 similarity matrix
for ax, seq_id, title in zip(axes, ["sequence_4", "sequence_1"],
                              ["Sequence 4 (Perfect GNN — Plant Growth)",
                               "Sequence 1 (GNN Fails — DIY Procedural)"]):
    emb_path = EMBEDDINGS / f"{seq_id}.npy"
    if emb_path.exists():
        emb = np.load(emb_path)
        normed = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
        sim_mat = normed @ normed.T
        sns.heatmap(sim_mat, annot=True, fmt=".2f", cmap="viridis",
                    ax=ax, vmin=0.6, vmax=1.0,
                    xticklabels=[f"F{i+1}" for i in range(8)],
                    yticklabels=[f"F{i+1}" for i in range(8)])
        ax.set_title(title, fontsize=11)

plt.suptitle("Similarity Matrix Comparison: Perfect vs. Failed Ordering", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Radar / Spider chart — compare methods across all metric dimensions
import matplotlib.patches as mpatches

metrics = ["pairwise_order_accuracy", "normalized_kendall_agreement", "exact_match_accuracy"]
metric_labels_short = ["Pairwise Acc", "Norm. Kendall", "Exact Match"]

# Inversion: convert inversion_count to a 0–1 "correctness" (1 = no inversions)
method_stats = by_method.set_index("method")

fig, ax = plt.subplots(figsize=(8, 5))
x_pos = np.arange(len(metrics))
width = 0.25

for i, (method, label) in enumerate(zip(method_order, method_labels)):
    if method not in method_stats.index:
        continue
    vals = [method_stats.loc[method, m] for m in metrics]
    bars = ax.bar(x_pos + i * width, vals, width, label=label,
                  color=list(PALETTE.values())[i], alpha=0.88)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x_pos + width)
ax.set_xticklabels(metric_labels_short)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.15)
ax.set_title("All Metrics — Method Comparison (Averaged Over 7 Sequences)")
ax.legend()
ax.axhline(0.5, ls="--", color="#aaa", alpha=0.6)
plt.tight_layout()
plt.show()

---
## 10. Key Findings

### What Works
1. **Greedy Nearest Neighbor significantly beats random**: 61.2% vs 50% pairwise accuracy (+11.2pp), with Kendall τ of +0.22. The visual smoothness assumption holds enough to be useful.
2. **Plant Growth sequences have the highest temporal contrast scores** (0.15–0.17 vs 0.04–0.07 for DIY/Coffee). Visual change is more consistent and monotonic in time-lapse biology content.
3. **Endpoint frames are meaningfully distinct**: true first and last frames tend to sit farther from the embedding centroid, suggesting an anchor-finding strategy has potential.
4. **Path scores correlate loosely with correctness**: GNN finds higher-scoring paths than Continuity, which may confirm that adjacency is the dominant ordering signal.

### What Doesn't Work
1. **Continuity exhaustive search performs worse than random** (29.1%): penalising non-smooth jumps in 8-frame sequences appears to over-constrain the search and produce reversed orderings.
2. **Forward/reverse disambiguation fails completely**: all 7 sequences have symmetric continuity scores — the embedding space cannot distinguish forward from backward temporal flow without text grounding.
3. **Procedural/DIY sequences are much harder**: higher frame similarity variance and less monotonic visual change make GNN unreliable.
4. **No method achieves > 61% pairwise accuracy**: the low temporal contrast scores (mean TCS ≈ 0.10) reflect an inherent difficulty in this embedding space for this content.

### Statistical Reliability
With only N=7 sequences, conclusions should be treated as directional hypotheses rather than statistically rigorous findings. Expanding to 50+ sequences per domain is needed for significance.

---
## 11. Future Optimizations & Research Directions

Based on the diagnostics, six concrete improvement paths are identified:

---

### 11.1 Better Embeddings

| Approach | Expected Gain | Effort |
|---|---|---|
| Switch to **ViT-L/14 OpenCLIP** (laion2b_s32b) | Higher TCS scores; richer visual features | Medium |
| **DINOv2 features** instead of CLIP | Scene-level geometry often better for temporal tasks | Medium |
| **VideoMAE / TimeSformer** video-native models | Directly encode temporal structure | High |
| **Fine-tune on temporal pairs** (supervised contrastive) | Could dramatically increase TCS | High |

The current TCS (mean ≈ 0.10) is very low — even the best sequence (Seq 2, 0.17) is barely above 0. Larger / temporally-aware encoders should produce better-separated embeddings.

---

### 11.2 Direction-Aware Ordering with Text Grounding

The forward/reverse tie was universal. The `build_directional_evidence()` function exists but was not activated in the primary experiments (`w_dir = 0`). Enabling it with captioned sequences allows CLIP's language-vision alignment to break the symmetry:

```python
# Currently inactive — enable direction scoring:
weights = ContinuityScoreWeights(adjacency=1.0, continuity=1.0, direction=0.75, endpoint=0.5)
```

**Hypothesis**: text prompts (`"start of building a house"`, `"end of building a house"`) will correctly orient ambiguous sequences.

---

### 11.3 Graph-Based Global Ordering (TSP / Beam Search)

The greedy path is a heuristic. A proper **Traveling Salesman Problem (TSP)** solver on the cosine similarity graph will find the globally optimal smooth path:

- Use **Lin-Kernighan** or **OR-Tools** for small n (≤ 20 frames)
- Use **beam search** with width k for medium n
- **Christofides algorithm** provides a guaranteed 3/2-approximation

For n=8 frames, the exhaustive search (40K permutations) is already tractable — the key issue is the *objective function*, not the search strategy.

---

### 11.4 Learned Pairwise Ordering (Ranking Models)

Rather than an unsupervised scoring function, train a **pairwise classifier** that predicts whether frame A comes before frame B:

```
Input: [embed(A), embed(B)] → Binary label: P(A before B)
```

This converts the ordering problem into a **learning-to-rank** task. Once a pairwise predictor is trained:
- Use **topological sort** or **rank aggregation** (Kemeny-optimal) to recover a global order
- Models can be pre-trained on large video datasets (Kinetics, HowTo100M) and zero-shot transferred

---

### 11.5 Expand and Balance the Dataset

The current 7-sequence benchmark has several limitations:
- All sequences have exactly 8 frames — testing with variable lengths (5–20 frames) is essential
- Heavy DIY/procedural bias (5 of 7 sequences are procedural or plant growth)
- No inter-sequence baselines or held-out test splits

**Recommended dataset expansion**:
- 50+ sequences across 10+ categories (cooking, construction, nature, medical, sports)
- 3-way split: train / val / test with stratification by difficulty and sequence type
- Include **distractor frames** (frames from different sequences) to test robustness

---

### 11.6 Ensemble & Hybrid Methods

Individual methods have complementary failure modes:
- GNN struggles with sequences where the embedding space is non-monotonic
- Continuity exhaustive search fails globally but might succeed locally

**Proposed hybrid**: 
1. Use GNN to generate k candidate paths from different start nodes
2. Re-rank candidates using the continuity + direction + endpoint composite score
3. Apply forward/reverse disambiguation with text grounding as final tiebreaker

This keeps the GNN's global path structure while using continuity as a local refinement oracle.

---

### 11.7 Evaluation Improvements

- Add **Spearman ρ** and **Average Displacement Error** (ADE) metrics
- Report **bootstrap confidence intervals** over N sequences
- Add **human baseline** (MTurk / expert annotation) as upper bound
- Track **partial credit**: how many frames are placed within ±1 position?

In [ ]:
# Visualise: hypothetical performance roadmap
import matplotlib.patches as mpatches

roadmap = {
    "Current\nGreedy NN": 0.612,
    "+ Direction\nAware (text)": 0.68,
    "+ TSP / Beam\nSearch": 0.70,
    "+ ViT-L/14\nEmbeddings": 0.76,
    "+ Pairwise\nRanking Model": 0.82,
    "+ Full\nDataset + FT": 0.88,
}

fig, ax = plt.subplots(figsize=(13, 5))
colors = ["#1976d2" if i == 0 else ("#4caf50" if v < 0.85 else "#ff9800")
          for i, v in enumerate(roadmap.values())]
bars = ax.barh(list(roadmap.keys()), list(roadmap.values()),
               color=colors, height=0.55, alpha=0.9)

for bar, val in zip(bars, roadmap.values()):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.1%}", va="center", fontsize=11, fontweight="bold")

ax.axvline(0.5, ls="--", color="#aaa", alpha=0.7, label="Random baseline")
ax.axvline(0.612, ls=":", color="#1976d2", alpha=0.5, label="Current GNN")
ax.set_xlim(0.3, 1.0)
ax.set_xlabel("Pairwise Order Accuracy")
ax.set_title("Estimated Performance Roadmap — Projected Gains from Future Work",
             fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print("Note: projected values are illustrative estimates based on typical gains")
print("reported in related works; actual gains will depend on implementation quality.")

---
## 12. Appendix: Reproducibility

### Running the Full Pipeline

```bash
# 1. Install dependencies
pip install -e ".[dev]"

# 2. Embed all sequences (cached in Data/embeddings/openclip/)
python embedder.py

# 3. Run similarity analysis
python scripts/similarity_matrices.py

# 4. Run ordering benchmark
python evaluate_ordering.py

# 5. Run full diagnostics
python scripts/run_diagnostics.py
```

### Package Structure

```
src/chronologic/
├── ordering/
│   ├── nearest_neighbor.py    # Greedy path + adjacency scoring
│   ├── continuity.py          # Exhaustive continuity search + ablations
│   ├── random_baseline.py     # Seeded random permutation
│   ├── reverse_disambiguation.py  # Forward/reverse tiebreaking
│   └── text_direction.py      # CLIP text-direction scoring
├── evaluation/
│   ├── metrics.py             # Exact match, pairwise, Kendall τ
│   └── runner.py              # Full benchmark orchestration
└── analysis/
    ├── alignment.py
    ├── endpoint_analysis.py
    ├── error_taxonomy.py
    ├── forward_reverse.py
    ├── pairwise_errors.py
    └── trajectory.py
```

### Hyperparameters

| Parameter | Value |
|---|---|
| Encoder | OpenCLIP RN50 (openai weights) |
| Embedding dim | 1024 |
| Similarity | Cosine |
| Near-gap | 1 (adjacent frames) |
| Far-gap | 4 (distant frames) |
| Random seeds | Seq 1–7: 42, 43, 44, 45, 46, 47, 48 |
| Continuity weights | adj=1.0, cont=1.0, dir=0.0, ep=0.0 |
| Continuity ablation w_dir | 0.75 |
| Continuity ablation w_ep | 0.75 |

In [ ]:
# Final summary printout
print("=" * 65)
print("CHRONOLOGIC — TEMPORAL ORDERING BENCHMARK SUMMARY")
print("=" * 65)
print(f"Sequences evaluated : 7")
print(f"Frames per sequence : 8")
print(f"Methods compared   : Random, Greedy NN, Continuity Exhaustive")
print()
for _, row in by_method.iterrows():
    m = row["method"].replace("_", " ").title()
    print(f"  {m:<35} Pairwise={row['pairwise_order_accuracy']:.3f}  τ={row['kendall_tau']:+.3f}")
print()
print("Best method    : Greedy Nearest Neighbor")
print("Primary finding: Forward/reverse ambiguity is a universal challenge")
print("Next steps     : Text-grounded direction scoring + larger encoder")
print("=" * 65)